In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import joblib

# Load
df = pd.read_csv('../dataset/AmesHousing.csv')


In [ ]:
def clean_data(df):
    # 1. Standardize names and handle basic IDs
    df.columns = df.columns.str.strip()
    to_drop = ['Order', 'PID']
    df = df.drop(columns=[c for c in to_drop if c in df.columns])

    # 2. Fill Missing Numerical Values (Crucial for Linear Regression)
    num_cols = df.select_dtypes(include=['number']).columns
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    
    # 3. Feature Engineering (The Big Ones)
    # Using 'Total Bsmt SF', '1st Flr SF', etc. from the specific dataset
    df['TotalSF'] = df['Total Bsmt SF'] + df['1st Flr SF'] + df['2nd Flr SF']

    # 4. Handle Categorical Values
    # We will pick the 5 most influential categorical columns to keep
    top_cats = ['Neighborhood', 'MS Zoning', 'Foundation', 'Kitchen Qual', 'House Style']
    
    # Fill NaNs in these specific columns so get_dummies works correctly
    for col in top_cats:
        if col in df.columns:
            df[col] = df[col].fillna('None')
    
    # One-Hot Encode only these top categories
    df = pd.get_dummies(df, columns=[c for c in top_cats if c in df.columns], drop_first=True)

    # 5. Drop all remaining non-numeric columns
    df = df.select_dtypes(include=['number', 'bool']) 
    
    # Convert booleans (from get_dummies) to 1/0 for the models
    for col in df.select_dtypes(include=['bool']).columns:
        df[col] = df[col].astype(int)

    return df

In [9]:
df = clean_data(df)

In [ ]:

# Prepare the data
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Models
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1)
}

# Training & Evaluation Loop
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
    print(f"{name} MAPE: {mape:.2f}%")
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    print(f"{name} RMSE: ${rmse:,.2f}")

LinearRegression MAPE: 9.82%
LinearRegression RMSE: $31,541.60
RandomForest MAPE: 8.17%
RandomForest RMSE: $25,461.24
XGBoost MAPE: 7.73%
XGBoost RMSE: $23,656.86


In [11]:
# Save the XGBoost model because it has the lowest MAPE and RMSE
if name == "XGBoost":
    joblib.dump(model, '../models/housing_model.pkl')
    joblib.dump(X.columns.tolist(), '../models/features.pkl')